In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from sklearn.preprocessing import StandardScaler
from PIL import Image
from tqdm import tqdm

In [ ]:
# Cihaz ayarı
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Veri Yolları
BASE_DIR = '/kaggle/input/competitions/planttraits2024'
TRAIN_CSV = os.path.join(BASE_DIR, 'train.csv')
TRAIN_IMG_DIR = os.path.join(BASE_DIR, 'train_images')

df_train = pd.read_csv(TRAIN_CSV)
TARGET_COLS = ['X4_mean', 'X11_mean', 'X18_mean', 'X26_mean', 'X50_mean', 'X3112_mean']

# Hedefleri Ölçeklendirme (RMSE patlamasını önlemek için)
target_scaler = StandardScaler()
df_train[TARGET_COLS] = target_scaler.fit_transform(df_train[TARGET_COLS])

print(f"Veri yüklendi ve ölçeklendirildi! Toplam örnek: {len(df_train)}")

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class MultimodalPlantDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform
        self.targets = TARGET_COLS
        
        all_other_cols = [col for col in df.columns if col not in ['id'] + self.targets]
        numeric_cols = df[all_other_cols].select_dtypes(include=[np.number]).columns.tolist()
        self.tabular_cols = numeric_cols[:31] 

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_id = str(int(self.df.iloc[idx]['id']))
        img_path = os.path.join(self.img_dir, f"{img_id}.jpeg")
            
        try:
            image = Image.open(img_path).convert("RGB")
        except Exception:
            image = Image.new('RGB', (224, 224))

        if self.transform:
            image = self.transform(image)

        tabular_data = torch.tensor(self.df.iloc[idx][self.tabular_cols].values.astype('float32'))
        labels = torch.tensor(self.df.iloc[idx][self.targets].values.astype('float32'))

        return image, tabular_data, labels

full_dataset = MultimodalPlantDataset(df=df_train, img_dir=TRAIN_IMG_DIR, transform=train_transforms)
# drop_last=True ekliyoruz ki BatchNorm son adımda çökmesin!
full_loader = DataLoader(full_dataset, batch_size=32, shuffle=True, num_workers=2, drop_last=True)
print("Veri Taşıyıcı (DataLoader) hazır!")

In [ ]:
class ModelB_CrossAttention(nn.Module):
    def __init__(self, num_tabular_features=31, embed_dim=256, num_heads=8, num_targets=6):
        super(ModelB_CrossAttention, self).__init__()
        
        # 1. Görsel Özellik Çıkarıcı (Omurga)
        weights = EfficientNet_B0_Weights.DEFAULT
        visual_model = efficientnet_b0(weights=weights)
        # Sadece özellikleri (feature map) almak için son iki katmanı (pooling ve classifier) atıyoruz.
        # Bu bize [batch, 1280, 7, 7] boyutunda uzamsal (spatial) bir harita verecek.
        self.visual_extractor = nn.Sequential(*list(visual_model.children())[:-2])
        
        # Görsel boyutunu (1280) ortak dikkat boyutuna (embed_dim = 256) düşür
        self.visual_proj = nn.Linear(1280, embed_dim)
        
        # 2. Tablosal Veri Gömme (Tabular Embedding)
        # 31 değişkeni ortak dikkat boyutuna (embed_dim = 256) çıkar
        self.tabular_proj = nn.Sequential(
            nn.Linear(num_tabular_features, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Linear(128, embed_dim)
        )
        
        # 3. Çapraz Dikkat Katmanı (Cross-Attention)
        # Tablosal veri "Soru/Rehber" (Query), Görsel veri "Cevap" (Key/Value) olacak.
        self.cross_attention = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, batch_first=True)
        
        # 4. Regresyon Başlığı (Sonuçları Üretme)
        self.regression_head = nn.Sequential(
            nn.Linear(embed_dim * 2, 128), # Tablo + Dikkat Edilmiş Görsel
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_targets)
        )

    def forward(self, images, tabular_data):
        batch_size = images.size(0)
        
        # --- A. GÖRSEL İŞLEME ---
        # Çıktı: [Batch, 1280, 7, 7]
        v_features = self.visual_extractor(images) 
        
        # [Batch, 1280, 7, 7] -> [Batch, 1280, 49] -> [Batch, 49, 1280] (49 adet görüntü yaması/patch)
        v_features = v_features.view(batch_size, 1280, -1).permute(0, 2, 1) 
        
        # Görsel yamaları ortak boyuta getir: [Batch, 49, 256]
        v_proj = self.visual_proj(v_features) 
        
        # --- B. TABLOSAL İŞLEME ---
        # Tablosal veriyi boyutlandır: [Batch, 256]
        t_proj = self.tabular_proj(tabular_data)
        
        # Attention katmanı 3 boyutlu veri bekler, bu yüzden araya sahte bir dizi boyutu ekliyoruz: [Batch, 1, 256]
        t_proj_unsq = t_proj.unsqueeze(1) 
        
        # --- C. ÇAPRAZ DİKKAT (SİHRİN OLDUĞU YER) ---
        # Query: Tablo (Rehber) | Key & Value: Görsel (Harita)
        # Tablodaki iklim şartlarına göre görüntünün o anki 49 parçasından hangilerine odaklanması gerektiğini seçer.
        attn_output, _ = self.cross_attention(query=t_proj_unsq, key=v_proj, value=v_proj)
        
        # Çıktıyı tekrar 2 boyuta düzleştir: [Batch, 256]
        attn_output = attn_output.squeeze(1) 
        
        # --- D. FÜZYON VE TAHMİN ---
        # Orijinal tablo verisiyle, tablonun dikkat ederek süzdüğü görsel veriyi birleştir
        fused = torch.cat((t_proj, attn_output), dim=1) # [Batch, 512]
        
        # 6 Ekolojik tahmini yap
        outputs = self.regression_head(fused)
        
        return outputs

# Modeli test etmek için cihaz ayarı
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_b = ModelB_CrossAttention().to(device)
print(f"Model B (Çapraz Dikkat) başarıyla kuruldu ve {device} cihazına taşındı!")

criterion = nn.MSELoss()
optimizer = optim.AdamW(model_b.parameters(), lr=1e-3, weight_decay=1e-4)

EPOCHS = 12
best_rmse = float('inf')
MODEL_SAVE_PATH = '/kaggle/working/cross_attention_best.pth'


for epoch in range(EPOCHS):
    model_b.train()
    running_loss = 0.0
    
    progress_bar = tqdm(full_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    
    for images, tab_data, labels in progress_bar:
        images = images.to(device)
        tab_data = tab_data.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        outputs = model_b(images, tab_data)
        
        loss = torch.sqrt(criterion(outputs, labels))
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        progress_bar.set_postfix(RMSE=f"{loss.item():.4f}")
        
    epoch_loss = running_loss / len(full_loader)
    print(f"Epoch {epoch+1} Özeti | Ortalama RMSE: {epoch_loss:.4f}")
    
    if epoch_loss < best_rmse:
        print(f"Yeni en iyi skor! Model kaydediliyor... ({best_rmse:.4f} -> {epoch_loss:.4f})\n")
        best_rmse = epoch_loss
        torch.save(model_b.state_dict(), MODEL_SAVE_PATH)
    else:
        print("\n")

print(f"Eğitim tamamlandı! En iyi model şuraya kaydedildi: {MODEL_SAVE_PATH}")